# Taller — Reconocimiento de Acciones Simples con Detección de Postura

Sistema de reconocimiento de acciones corporales en tiempo real usando la
**Tasks API de MediaPipe** (`PoseLandmarker`) y **OpenCV**.

**Acciones reconocidas:** persona sentada · brazos levantados · caminando

---

### Nota importante sobre la versión de la API

A partir de MediaPipe `0.10.31` se eliminó la *Legacy Solutions API*
(`mp.solutions.pose`). Este notebook usa la **Tasks API** actual, que:

- Requiere descargar un archivo de modelo `.task` por separado.
- Usa `PoseLandmarker.create_from_options(...)` para crear el detector.
- Convierte cada frame a un objeto `mp.Image` antes de procesarlo.
- Devuelve el resultado en `.pose_landmarks` (lista de poses detectadas).

### Estructura del notebook
1. Verificación del entorno
2. Descarga del modelo `.task`
3. Imports e inicialización del PoseLandmarker
4. Funciones auxiliares — ángulos y distancias
5. Dibujo del esqueleto sobre el frame
6. Lógica de clasificación de acciones
7. Retroalimentación multimodal (visual y sonora)
8. Bucle principal de captura en tiempo real
9. Liberación de recursos

> El video se abre en una **ventana externa de OpenCV**. Para detener la captura
> pulsa la tecla **`q`** sobre esa ventana.


## 1. Verificación del entorno

Confirma que el kernel es Python 3.11 del entorno `taller-posturas` y que las
librerías responden. La versión de MediaPipe debe ser `0.10.x`.


In [17]:
import sys
import cv2
import mediapipe as mp
import numpy as np

print("Python   :", sys.version.split()[0])
print("OpenCV   :", cv2.__version__)
print("MediaPipe:", mp.__version__)
print("NumPy    :", np.__version__)

try:
    import pygame
    print("pygame   :", pygame.version.ver, "(sonido disponible)")
    SONIDO_DISPONIBLE = True
except Exception:
    print("pygame no disponible -> se usara solo retroalimentacion visual")
    SONIDO_DISPONIBLE = False


Python   : 3.11.15
OpenCV   : 4.13.0
MediaPipe: 0.10.35
NumPy    : 2.4.6
pygame   : 2.6.1 (sonido disponible)


## 2. Descarga del modelo `.task`

La Tasks API **no incluye el modelo dentro de la librería**: hay que descargar
un archivo `.task`. MediaPipe ofrece tres variantes con la misma salida de 33
landmarks, que se diferencian en precisión y velocidad:

| Modelo | Tamaño aprox. | Uso |
|---|---|---|
| `lite`  | ~3 MB  | El más rápido |
| `full`  | ~6 MB  | Equilibrado (recomendado) |
| `heavy` | ~30 MB | El más preciso, más lento |

Para este taller, **`full`** da buena precisión sin sacrificar el tiempo real.

La celda siguiente intenta descargar el modelo automáticamente. **Si no tienes
conexión o falla**, descarga el archivo manualmente desde esta URL y guárdalo
en la misma carpeta del notebook con el nombre `pose_landmarker_full.task`:

`https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task`


In [18]:
import os
import urllib.request

MODELO = "pose_landmarker_full.task"
URL_MODELO = ("https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
              "pose_landmarker_full/float16/latest/pose_landmarker_full.task")

if os.path.exists(MODELO):
    tam = os.path.getsize(MODELO) / 1024
    print(f"Modelo ya presente: {MODELO} ({tam:.0f} KB)")
else:
    print("Descargando modelo...")
    try:
        urllib.request.urlretrieve(URL_MODELO, MODELO)
        tam = os.path.getsize(MODELO) / 1024
        print(f"Modelo descargado: {MODELO} ({tam:.0f} KB)")
    except Exception as e:
        print("ERROR al descargar:", e)
        print("Descarga el archivo manualmente (ver instrucciones arriba)")


Modelo ya presente: pose_landmarker_full.task (9178 KB)


## 3. Imports e inicialización del PoseLandmarker

La Tasks API se importa desde `mediapipe.tasks`. Se crea un objeto `PoseLandmarker`
con `create_from_options`, indicando:

- `base_options` — ruta al modelo `.task`.
- `running_mode = VIDEO` — modo para fotogramas de video. En este modo el detector
  bloquea el hilo hasta terminar cada frame, lo que simplifica el bucle (a diferencia
  del modo `LIVE_STREAM`, que es asíncrono con callbacks).
- umbrales de confianza para detección y seguimiento.

El detector se crea **una sola vez** y se reutiliza durante toda la sesión.


In [19]:
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

BaseOptions          = mp_python.BaseOptions
PoseLandmarker       = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions
VisionRunningMode    = vision.RunningMode

opciones = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODELO),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

# detector reutilizable durante toda la sesion
landmarker = PoseLandmarker.create_from_options(opciones)
print("PoseLandmarker inicializado (modo VIDEO) — 33 landmarks por pose")


PoseLandmarker inicializado (modo VIDEO) — 33 landmarks por pose


## 4. Funciones auxiliares — ángulos y distancias

En la Tasks API, el resultado de detección trae `resultado.pose_landmarks`, que
es una **lista de poses**; cada pose es a su vez una lista de 33 landmarks.
Cada landmark tiene atributos `x`, `y`, `z`, `visibility` y `presence`.

Las coordenadas `x` e `y` están **normalizadas** en `[0, 1]` (0 = arriba para `y`).


In [20]:
def obtener_punto(landmarks, indice):
    """Devuelve (x, y) normalizados de un landmark dado su indice."""
    lm = landmarks[indice]
    return np.array([lm.x, lm.y])


def calcular_angulo(a, b, c):
    """Angulo en grados en el vertice b, formado por los puntos a-b-c."""
    a, b, c = np.array(a), np.array(b), np.array(c)
    radianes = np.arctan2(c[1] - b[1], c[0] - b[0]) - \
               np.arctan2(a[1] - b[1], a[0] - b[0])
    angulo = np.abs(np.degrees(radianes))
    if angulo > 180.0:
        angulo = 360.0 - angulo
    return angulo


def distancia(p1, p2):
    """Distancia euclidiana entre dos puntos (x, y)."""
    return float(np.linalg.norm(np.array(p1) - np.array(p2)))


print("Angulo recto esperado ~90:", round(calcular_angulo([0, 0], [0, 1], [1, 1]), 1))
print("Distancia esperada 1.0  :", round(distancia([0, 0], [1, 0]), 1))


Angulo recto esperado ~90: 90.0
Distancia esperada 1.0  : 1.0


## 5. Dibujo del esqueleto sobre el frame

La Tasks API ya no usa las utilidades antiguas de dibujo de la misma forma. Aquí
se define una función propia que dibuja los **33 puntos** y las **conexiones
esqueléticas** directamente con OpenCV. Esto da control total sobre el estilo y
no depende de APIs que puedan cambiar.

Las conexiones se basan en el esquema estándar de 33 landmarks de MediaPipe Pose.


In [21]:
# Indices de los 33 landmarks (mismos que la API antigua)
NARIZ = 0
HOMBRO_IZQ, HOMBRO_DER = 11, 12
CODO_IZQ, CODO_DER = 13, 14
MUNECA_IZQ, MUNECA_DER = 15, 16
CADERA_IZQ, CADERA_DER = 23, 24
RODILLA_IZQ, RODILLA_DER = 25, 26
TOBILLO_IZQ, TOBILLO_DER = 27, 28

# Conexiones esqueleticas principales (pares de indices)
CONEXIONES = [
    (HOMBRO_IZQ, HOMBRO_DER), (HOMBRO_IZQ, CODO_IZQ), (CODO_IZQ, MUNECA_IZQ),
    (HOMBRO_DER, CODO_DER), (CODO_DER, MUNECA_DER),
    (HOMBRO_IZQ, CADERA_IZQ), (HOMBRO_DER, CADERA_DER),
    (CADERA_IZQ, CADERA_DER),
    (CADERA_IZQ, RODILLA_IZQ), (RODILLA_IZQ, TOBILLO_IZQ),
    (CADERA_DER, RODILLA_DER), (RODILLA_DER, TOBILLO_DER),
]


def dibujar_esqueleto(frame, landmarks):
    """Dibuja puntos y conexiones de los landmarks sobre el frame (BGR)."""
    h, w = frame.shape[:2]

    # convertir coordenadas normalizadas a pixeles
    puntos = [(int(lm.x * w), int(lm.y * h)) for lm in landmarks]

    # conexiones (lineas)
    for i, j in CONEXIONES:
        cv2.line(frame, puntos[i], puntos[j], (245, 245, 245), 2)

    # puntos (circulos) — solo si el landmark es suficientemente visible
    for idx, lm in enumerate(landmarks):
        if lm.visibility > 0.5:
            cv2.circle(frame, puntos[idx], 4, (0, 0, 255), -1)

    return frame


## 6. Lógica de clasificación de acciones

Cada acción se reconoce con reglas geométricas. Recuerda: en coordenada `y`, un
valor **menor** está más arriba en la imagen.

| Acción | Regla |
|---|---|
| **Brazos levantados** | ambas muñecas por encima de la nariz |
| **Sentado** | rodillas flexionadas (ángulo cadera–rodilla–tobillo < 120°) y caderas cerca de la altura de las rodillas |
| **Caminando** | alternancia: cruces en la diferencia de altura de los tobillos entre frames |

Caminar necesita memoria entre frames, por eso se usa el diccionario de estado
`estado_caminar`.


In [22]:
# Estado persistente para detectar alternancia de pies (caminar)
estado_caminar = {"dif_previa": 0.0, "cambios": 0, "frames": 0}


def clasificar_accion(landmarks):
    """Devuelve el nombre de la accion detectada a partir de los 33 landmarks."""

    nariz       = obtener_punto(landmarks, NARIZ)
    muneca_izq  = obtener_punto(landmarks, MUNECA_IZQ)
    muneca_der  = obtener_punto(landmarks, MUNECA_DER)
    cadera_izq  = obtener_punto(landmarks, CADERA_IZQ)
    cadera_der  = obtener_punto(landmarks, CADERA_DER)
    rodilla_izq = obtener_punto(landmarks, RODILLA_IZQ)
    rodilla_der = obtener_punto(landmarks, RODILLA_DER)
    tobillo_izq = obtener_punto(landmarks, TOBILLO_IZQ)
    tobillo_der = obtener_punto(landmarks, TOBILLO_DER)

    # --- 1. BRAZOS LEVANTADOS ---
    if muneca_izq[1] < nariz[1] and muneca_der[1] < nariz[1]:
        return "Brazos levantados"

    # --- 2. SENTADO ---
    ang_rodilla_izq = calcular_angulo(cadera_izq, rodilla_izq, tobillo_izq)
    ang_rodilla_der = calcular_angulo(cadera_der, rodilla_der, tobillo_der)
    rodillas_flexionadas = ang_rodilla_izq < 120 and ang_rodilla_der < 120
    caderas_bajas = (cadera_izq[1] > rodilla_izq[1] - 0.10 and
                     cadera_der[1] > rodilla_der[1] - 0.10)
    if rodillas_flexionadas and caderas_bajas:
        return "Sentado"

    # --- 3. CAMINANDO ---
    dif_actual = tobillo_izq[1] - tobillo_der[1]
    estado_caminar["frames"] += 1
    if estado_caminar["dif_previa"] * dif_actual < 0 and abs(dif_actual) > 0.03:
        estado_caminar["cambios"] += 1
    estado_caminar["dif_previa"] = dif_actual
    if estado_caminar["frames"] >= 15:
        caminando = estado_caminar["cambios"] >= 2
        estado_caminar["cambios"] = 0
        estado_caminar["frames"] = 0
        if caminando:
            clasificar_accion.ultimo_caminar = 20
    if getattr(clasificar_accion, "ultimo_caminar", 0) > 0:
        clasificar_accion.ultimo_caminar -= 1
        return "Caminando"

    return "Sin accion definida"


clasificar_accion.ultimo_caminar = 0
print("Logica de clasificacion lista")


Logica de clasificacion lista


## 7. Retroalimentación multimodal

- **Visual:** barra superior con la acción detectada y panel con los ángulos de rodilla.
- **Sonora (opcional):** un *ding* cuando **cambia** la acción detectada. Coloca un
  archivo `ding.wav` en la carpeta del notebook para activarlo; si no existe, el
  sistema funciona igual solo con la parte visual.


In [23]:
sonido_ding = None
if SONIDO_DISPONIBLE:
    try:
        pygame.mixer.init()
        sonido_ding = pygame.mixer.Sound("ding.wav")
        print("Sonido 'ding.wav' cargado correctamente")
    except Exception:
        print("No se encontro 'ding.wav' -> solo retroalimentacion visual")
        sonido_ding = None


def reproducir_ding():
    if sonido_ding is not None:
        sonido_ding.play()


COLORES = {
    "Brazos levantados":   (0, 200, 0),
    "Sentado":             (0, 165, 255),
    "Caminando":           (255, 120, 0),
    "Sin accion definida": (160, 160, 160),
    "Sin persona detectada": (90, 90, 90),
}


def dibujar_interfaz(frame, accion, landmarks):
    """Dibuja la etiqueta de accion y el panel de angulos sobre el frame."""
    color = COLORES.get(accion, (200, 200, 200))
    h, w = frame.shape[:2]

    cv2.rectangle(frame, (0, 0), (w, 60), color, -1)
    cv2.putText(frame, f"Accion: {accion}", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)

    if landmarks is not None:
        ai = calcular_angulo(obtener_punto(landmarks, CADERA_IZQ),
                             obtener_punto(landmarks, RODILLA_IZQ),
                             obtener_punto(landmarks, TOBILLO_IZQ))
        ad = calcular_angulo(obtener_punto(landmarks, CADERA_DER),
                             obtener_punto(landmarks, RODILLA_DER),
                             obtener_punto(landmarks, TOBILLO_DER))
        cv2.putText(frame, f"Rodilla izq: {ai:5.1f}", (20, h - 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame, f"Rodilla der: {ad:5.1f}", (20, h - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.putText(frame, "Pulsa 'q' para salir", (w - 250, h - 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return frame


No se encontro 'ding.wav' -> solo retroalimentacion visual


## 8. Bucle principal de captura en tiempo real

Esta es la celda central. Al ejecutarla se abre una **ventana externa** con el video.

Flujo por cada frame, según la Tasks API:
1. Leer el frame con OpenCV (formato BGR).
2. Convertir a RGB y envolverlo en un objeto `mp.Image`.
3. Llamar a `landmarker.detect_for_video(mp_image, timestamp_ms)`. El modo VIDEO
   **exige un timestamp en milisegundos**, creciente en cada llamada.
4. Leer `resultado.pose_landmarks` — lista de poses; usamos la primera.
5. Dibujar esqueleto, clasificar la acción y mostrar la interfaz.

Para terminar, pulsa **`q`** sobre la ventana de video.

> Si el bucle se cuelga, pulsa **■ (interrumpir kernel)** y ejecuta la celda 9.


In [ ]:
import time

cap = cv2.VideoCapture(0)   # cambia a 1 si tienes varias camaras

if not cap.isOpened():
    print("ERROR: no se pudo abrir la webcam. Revisa permisos de camara.")
else:
    accion_previa = None
    t_inicio = time.time()
    print("Captura iniciada — pulsa 'q' en la ventana de video para salir")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("No se recibio frame de la camara")
            break

        frame = cv2.flip(frame, 1)  # efecto espejo

        # convertir frame BGR -> RGB -> objeto mp.Image
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

        # timestamp creciente en milisegundos (requerido en modo VIDEO)
        timestamp_ms = int((time.time() - t_inicio) * 1000)

        # deteccion con la Tasks API
        resultado = landmarker.detect_for_video(mp_image, timestamp_ms)

        accion = "Sin persona detectada"
        landmarks = None

        # pose_landmarks es una lista de poses; tomamos la primera
        if resultado.pose_landmarks:
            landmarks = resultado.pose_landmarks[0]
            frame = dibujar_esqueleto(frame, landmarks)
            accion = clasificar_accion(landmarks)

        frame = dibujar_interfaz(frame, accion, landmarks)

        # retroalimentacion sonora al cambiar de accion
        if (accion != accion_previa and accion in COLORES
                and accion not in ("Sin accion definida", "Sin persona detectada")):
            reproducir_ding()
            print("Accion detectada:", accion)
        accion_previa = accion

        cv2.imshow("Reconocimiento de Acciones - MediaPipe Tasks API", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("Captura finalizada")


Captura iniciada — pulsa 'q' en la ventana de video para salir
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Caminando
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Caminando
Accion detectada: Sentado
Accion detectada: Caminando
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Sentado
Accion detectada: Caminando
Accion detectada: Sentado
Accion detectada: S

## 9. Liberación de recursos

Ejecuta esta celda si detienes el bucle de forma abrupta. Suelta la cámara y
cierra las ventanas de OpenCV. Si quieres liberar también el detector (por
ejemplo, para recrearlo con otros parámetros), descomenta la última línea.


In [ ]:
try:
    cap.release()
except Exception:
    pass
cv2.destroyAllWindows()
for _ in range(4):
    cv2.waitKey(1)
print("Recursos liberados — camara y ventanas cerradas")

# Para cerrar tambien el detector (opcional):
# landmarker.close()


## Notas finales y posibles ajustes

- **Tasks API vs Legacy:** este notebook usa la API actual. Los puntos clave del
  cambio son: modelo `.task` descargado aparte, creación con `create_from_options`,
  conversión de frames a `mp.Image`, y resultado en `resultado.pose_landmarks`
  (una lista de poses).
- **Modo VIDEO:** se usó `detect_for_video` con timestamp en milisegundos. El
  timestamp debe ser **estrictamente creciente** entre llamadas, por eso se calcula
  a partir de `time.time()`.
- **Umbrales de las reglas:** los valores (`120°` para rodilla, `0.03` para
  alternancia de pies) son un punto de partida. Ajústalos en la celda 6 según tu
  cámara y reejecuta solo esa celda.
- **Calibración:** "sentado" funciona mejor con la cámara captando el cuerpo de
  lado o completo; "caminar" requiere que se vean los tobillos.
- **Sonido:** coloca un `ding.wav` corto en la carpeta del notebook para la
  retroalimentación sonora.
- **Entrega:** ejecuta las celdas en orden y guarda el notebook con las salidas
  visibles.
